# Fundamentos de Aprendizagem Profunda

## Estudo 2 — ECG5000: classificação com CNN, LSTM e GRU


Este estudo utiliza o conjunto de dados ECG5000, com sinais de eletrocardiogramas classificados em diferentes tipos de batimentos cardíacos. O notebook prepara os dados, treina e avalia três modelos: uma rede convolucional 1D (CNN) e duas redes recorrentes (LSTM e GRU). Os resultados são analisados a partir das métricas produzidas durante a execução.


## Bibliotecas e configuração

In [ ]:

import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, Callback, ModelCheckpoint
import pandas as pd
import matplotlib.pyplot as plt
# Resolve paths from either the project root or the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "notebooks").is_dir():
    raise FileNotFoundError("Open this project or its notebooks directory before running.")



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Dispositivo ativo: {device}", end="")
if device.type == "cuda":
    print(f" ({torch.cuda.get_device_name(0)})")
else:
    print()

In [ ]:
# SETUP
DATA_FOLDER = PROJECT_ROOT / "data" / "ecg5000"
RESULTS_FOLDER = PROJECT_ROOT / "results" / "ecg5000"
TRAIN_PATH  = os.path.join(DATA_FOLDER, "ECG5000_TRAIN.txt")
TEST_PATH   = os.path.join(DATA_FOLDER, "ECG5000_TEST.txt")

# Split Data
TEST_SIZE      = 0.15      # 15% para teste
VAL_SIZE_TOTAL = 0.15      # 15% do total para validação

# Seeds
SEED = 42

# Normalize
NORMALIZE = True           # True = aplicar StandardScaler


# Train
BATCH_SIZE   = 64
LEARNING_RATE = 1e-4
EPOCHS       = 100

# Modelo
NUM_CLASSES = 5
INPUT_SIZE  = 140

In [ ]:
# ===============================
# Callback para métricas
# ===============================

class MetricsLogger(Callback):
    def __init__(self):
        super().__init__()

        # Listas para armazenar métricas de treino e validação
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

    # Método chamado no final de cada epoch de treino
    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics                              # Métricas do Trainer
        self.train_loss.append(metrics.get("train_loss_epoch").item())  # Perda de validação
        self.train_acc.append(metrics.get("train_acc_epoch").item())    # Accuracy de train

    # Método chamado no final de cada epoch de validação
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics                              # Métricas do Trainer
        self.val_loss.append(metrics.get("val_loss_epoch").item())      # Perda de validação
        self.val_acc.append(metrics.get("val_acc_epoch").item())        # Exatidão de validação

# =========================================
# Função para converter métricas em DataFrame pandas.
# =========================================

def metrics_df(metrics_logger):

    # Encontrar o tamanho máximo entre todas as listas
    max_len = max(len(metrics_logger.train_loss),
                  len(metrics_logger.val_loss),
                  len(metrics_logger.train_acc),
                  len(metrics_logger.val_acc))
    
    # Função auxiliar para preencher listas com NaN.
    def pad_list(lst, length):
        return lst + [np.nan]*(length - len(lst))
    
    # Preencher listas
    train_loss = pad_list(metrics_logger.train_loss, max_len)
    val_loss   = pad_list(metrics_logger.val_loss, max_len)
    train_acc  = pad_list(metrics_logger.train_acc, max_len)
    val_acc    = pad_list(metrics_logger.val_acc, max_len)
    
    # Criar DataFrame
    df_metrics = pd.DataFrame({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc
    })

    # Definir o índice como epoch
    df_metrics.index.name = "epoch"
    
    return df_metrics

# ==========================================
# Função para salvar métricas em arquivo CSV
# ==========================================

def save_metrics_csv(metrics_logger, filename="defaultModel.csv", folder="results"):
    
    # Criar a pasta, caso não exista
    os.makedirs(folder, exist_ok=True)

    # Descobrir o número máximo de épocas registrada
    num_epochs = max(len(metrics_logger.train_loss),
                     len(metrics_logger.train_acc),
                     len(metrics_logger.val_loss),
                     len(metrics_logger.val_acc))
    
    # Função auxiliar para preencher listas menores com NaN
    def pad_list(lst):
        return lst + [np.nan] * (num_epochs - len(lst))
    
    # Criar DataFrame
    df_metrics = pd.DataFrame({
        "epoch": range(1, num_epochs + 1),
        "train_loss": pad_list(metrics_logger.train_loss),
        "train_acc": pad_list(metrics_logger.train_acc),
        "val_loss": pad_list(metrics_logger.val_loss),
        "val_acc": pad_list(metrics_logger.val_acc)
    })
    
    # Caminho completo do arquivo
    filepath = os.path.join(folder, filename)

    # Guardar CSV
    df_metrics.to_csv(filepath, index=False)
    print(f"CSV salvo com sucesso: {filepath}")

## Dados

O estudo utiliza o ECG5000, com segmentos de sinais de eletrocardiograma agrupados em cinco classes. Os ficheiros originais de treino e teste são reunidos antes de criar uma nova divisão para esta experiência.


### Carregamento

Carregam-se os dois ficheiros originais e reúnem-se os exemplos antes da divisão usada neste notebook.


In [ ]:
# Cada linha representa um ciclo de 140 pontos do batimento cardíaco, 
# sendo que a coluna 0 contém a label.
# Concatena-se, portanto, as 4500 amostras do dataset de teste com as 
# 500 amostras do dataset de treino, aplicando um shuffle no final 
# para misturar os exemplos.

train_df = pd.read_csv(TRAIN_PATH, header=None, sep=r"\s+")
test_df  = pd.read_csv(TEST_PATH, header=None, sep=r"\s+")

df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle total

### Análise

In [ ]:
# Após construir o DataFrame df, vamos verificar cada coluna
# para identificar qual contém a label (classe do batimento cardíaco).
# Contamos o número de valores únicos em cada coluna.

# Através deste processo conseguimos identificar e confirmar
# que a coluna 0 será a coluna que contém a label.

# Contagem de unicos
unique_counts = {col: df[col].nunique() for col in df.columns}

# Sort pelo numero de unicos
sorted_counts = dict(sorted(unique_counts.items(), key=lambda item: item[1]))

# Imprimir o resultado
for col, count in sorted_counts.items():
    print(f"Coluna {col}: {count} valores únicos")



In [ ]:
# Vamos aplicar uma contagem de classes para perceber o desbalanço dos dados.

# Podemos notar que:
# As classes majoritárias são a classe 1 e 2, com 2919 e 1767 exemplos respetivamente.
# As classes minoritárias são a classe 3, 4 e 5, com 96, 194 e 24 exemplos respetivamente.

# Isto prova que estamos perante um claro exemplo de desbalanço, 
# que terá de ser levado em conta no desenvolvimento e interpretação dos modelos futuros.

class_counts = df[0].value_counts().sort_index()
print("Número de exemplos por classe:")
print(class_counts)

### Preparação

Após reunir os ficheiros originais, o código reserva **15%** dos exemplos para teste e **15%** para validação; os **70%** restantes são usados no treino. A divisão é estratificada por classe. A normalização é ajustada apenas aos dados de treino e aplicada depois à validação e ao teste.


In [ ]:
# Separar features e labels
X = df.iloc[:, 1:].values.astype(np.float32)  # colunas 1 a 140
y = df.iloc[:, 0].values.astype(int)          # coluna 0

# Separar Teste (TEST_SIZE) e Restante
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Separar Validação (VAL_SIZE_TOTAL do total) a partir do restante
val_size = VAL_SIZE_TOTAL / (1 - TEST_SIZE)  # proporção relativa dentro do restante
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=val_size, random_state=SEED, stratify=y_rest
)

print("Shapes originais:")
print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test.shape)


if NORMALIZE:
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

y_train = torch.tensor(y_train - 1, dtype=torch.long)
y_val   = torch.tensor(y_val - 1, dtype=torch.long)
y_test  = torch.tensor(y_test - 1, dtype=torch.long)

def print_dataset_info(X, y, name):
    print(f"\n=== Dados de {name} ===")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("X dtype:", X.dtype)
    print("y dtype:", y.dtype)
    print("y min/max:", y.min().item(), "/", y.max().item())
    print("Exemplo X[0][:5]:", X[0][:5], "...")
    print("Exemplo y[:5]:", y[:5].tolist())

print_dataset_info(X_train, y_train, "Treino")
print_dataset_info(X_val, y_val, "Validação")
print_dataset_info(X_test, y_test, "Teste")

## Modelos

O estudo compara três modelos: uma rede convolucional 1D (CNN), que procura padrões locais na sequência, e duas redes recorrentes (LSTM e GRU), usadas para modelar dependências ao longo da sequência.


### 1D-CNN

In [ ]:
# Data Preparation para o CNN 
# Converter features para tensores PyTorch
# Adiciona a dimensão do canal [batch, 1, seq_len] necessária para Conv1D
X_train_cnn = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val_cnn   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
X_test_cnn  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

print("Tensores prontos para 1D-CNN:")
print("X_train_cnn:", X_train_cnn.shape)
print("X_val_cnn:", X_val_cnn.shape)
print("X_test_cnn:", X_test_cnn.shape)
print("y_train:", y_train.shape)


### Data Loaders 
cnn_train_dataset = TensorDataset(X_train_cnn, y_train)
cnn_val_dataset   = TensorDataset(X_val_cnn, y_val)
cnn_test_dataset  = TensorDataset(X_test_cnn, y_test)

cnn_train_loader = DataLoader(cnn_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
cnn_val_loader   = DataLoader(cnn_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
cnn_test_loader  = DataLoader(cnn_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Exemplo
cnn_batch_X, cnn_batch_y = next(iter(cnn_train_loader))
print("\nExemplo de batch:")
print("cnn_batch_X:", cnn_batch_X.shape)  # (batch_size, 1, seq_len)
print("cnn_batch_y:", cnn_batch_y.shape)  # (batch_size,)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, input_length=140, num_classes=5):
        super().__init__()
        
        # 1ª camada convolucional
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        # 2ª camada convolucional
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        # Flatten
        self.flatten = nn.Flatten()

        # DropOut
        self.dropout = nn.Dropout(0.2)

        # Fully connected
        conv_output_length = input_length // 4
        self.fc1 = nn.Linear(32 * conv_output_length, 32)

        # Output
        self.fc2 = nn.Linear(32, num_classes)

        # Inicialização dos pesos e biases
        self._initialize_weights()
    
    def forward(self, x):

        # 1ª convolução
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # 2ª convolução
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Achata o tensor para as FC
        x = self.flatten(x)

        # dropout
        x = self.dropout(x)

        # Primeira FC + ativação ReLU
        x = self.fc1(x)
        x = F.relu(x)

        # Camada final
        x = self.fc2(x)

        return x

    def _initialize_weights(self):
        
        # Inicialização dos pesos das camadas
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)


In [ ]:
class LitCNN(pl.LightningModule):
    def __init__(self, input_length=140, num_classes=5, lr=1e-3):
        super().__init__()

        # Modelo CNN
        self.model = CNNModel(input_length, num_classes)

        # Taxa de aprendizagem
        self.lr = lr

        # Função de perda para classificação multiclasse
        self.criterion = nn.CrossEntropyLoss()
    
    # Forward para inferência
    def forward(self, x):
        return self.model(x)

     # Passo de treino executado em cada batch
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
        return loss
    
    # Passo de validação
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("val_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("val_acc", acc, on_step=True, on_epoch=True, prog_bar=True)

    # Passo de teste
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("test_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("test_acc", acc, on_step=True, on_epoch=True, prog_bar=True)

    # Otimizador utilizado no treino
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


# EarlyStopping
cnn_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    verbose=True,
    mode="min"
)

# Checkpoint, guardar o melhor modelo com base na validação
cnn_checkpoint = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-cnn-model-v3",
    verbose=True
)

In [ ]:
# Instanciar modelo e logger
cnn_model = LitCNN(input_length=INPUT_SIZE, num_classes=NUM_CLASSES, lr=LEARNING_RATE)
cnn_metrics_logger = MetricsLogger()

# Trainer
cnn_trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="cpu",
    log_every_n_steps=10,
    deterministic=True,
    callbacks=[cnn_early_stop, cnn_metrics_logger, cnn_checkpoint]
)

# Treinar
cnn_trainer.fit(cnn_model, train_dataloaders=cnn_train_loader, val_dataloaders=cnn_val_loader)

In [ ]:
# Métricas em DataFrame
cnn_df_metrics = metrics_df(cnn_metrics_logger)
display(cnn_df_metrics)

#  Guardar CSV
save_metrics_csv(cnn_metrics_logger, filename="defaultModelCNN.csv", folder=RESULTS_FOLDER)

In [ ]:
# Teste final com melhor modelo
best_model_path = cnn_checkpoint.best_model_path
print("Best model saved at:", best_model_path)

cnn_test_results = cnn_trainer.test(
    dataloaders=cnn_test_loader,
    ckpt_path=best_model_path
)

print("\nResultados do Teste:", cnn_test_results)


### LSTM

In [ ]:
# Data Preparation for LSTM
# Converter features para tensores 3D [batch, seq_len, features]
X_train_lstm = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)  # (N, 140, 1)
X_val_lstm   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1)
X_test_lstm  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)

print("Tensores prontos para LSTM:")
print("X_train_lstm:", X_train_lstm.shape)
print("X_val_lstm:", X_val_lstm.shape)
print("X_test_lstm:", X_test_lstm.shape)
print("y_train:", y_train.shape)

# DataLoaders 
lstm_train_dataset = TensorDataset(X_train_lstm, y_train)
lstm_val_dataset   = TensorDataset(X_val_lstm, y_val)
lstm_test_dataset  = TensorDataset(X_test_lstm, y_test)

lstm_train_loader = DataLoader(lstm_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
lstm_val_loader   = DataLoader(lstm_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
lstm_test_loader  = DataLoader(lstm_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Exemplo de batch
lstm_batch_X, lstm_batch_y = next(iter(lstm_train_loader))
print("\nExemplo de batch:")
print("lstm_batch_X:", lstm_batch_X.shape)  # (batch_size, seq_len, features)
print("lstm_batch_y:", lstm_batch_y.shape)  # (batch_size,)


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, num_classes=5):
        super().__init__()

        hidden_size = 64 

        # Camadas LSTM 
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        # Classificador 
        self.fc1 = nn.Linear(hidden_size, 64)

        # Dropout
        self.dropout_fc = nn.Dropout(0.3)

        # Classificador
        self.fc2 = nn.Linear(64, 32)

        # Output
        self.fc3 = nn.Linear(32, num_classes)


    def forward(self, x):
        
        # LSTM 
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]

        # Classificador 
        x = F.relu(self.fc1(last_hidden))

        # Dropout
        x = self.dropout_fc(x)

        # Classificador 
        x = F.relu(self.fc2(x))

        # Output
        x = self.fc3(x)

        return x


In [ ]:
class LitLSTM(pl.LightningModule):
    def __init__(self, input_size=1, num_classes=5, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.criterion = nn.CrossEntropyLoss()
        self.model = LSTMModel(input_size=input_size, num_classes=num_classes)
    
    # Forward para inferência
    def forward(self, x):
        return self.model(x)
    
    # Passo de treino
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
        return loss
    
    # Passo de validação
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("val_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("val_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
    
    # Passo de teste
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("test_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("test_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
    
    # Otimizador utilizado no treino
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


# EarlyStopping
lstm_early_stop = EarlyStopping(
    monitor="val_loss",      
    patience=10,             
    verbose=True,
    mode="min"               
)

# Checkpoint, guardar o melhor modelo com base na validação
lstm_checkpoint = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-lstm-model",
    verbose=True
)

In [ ]:
# Instanciar modelo e logger
lstm_model = LitLSTM(input_size=1, num_classes=NUM_CLASSES, lr=LEARNING_RATE)
lstm_metrics_logger = MetricsLogger()

# Trainer
lstm_trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="cpu",
    log_every_n_steps=15,
    deterministic=True,
    callbacks=[lstm_early_stop, lstm_metrics_logger, lstm_checkpoint]
)

# Treinar
lstm_trainer.fit(lstm_model, train_dataloaders=lstm_train_loader, val_dataloaders=lstm_val_loader)

In [ ]:
# Métricas em DataFrame
lstm_df_metrics = metrics_df(lstm_metrics_logger)
display(lstm_df_metrics)

# Guardar CSV
save_metrics_csv(lstm_metrics_logger, filename="defaultModelLSTM.csv", folder=RESULTS_FOLDER)

In [ ]:
# Teste final com melhor modelo
best_model_path = lstm_checkpoint.best_model_path
print("Best LSTM model saved at:", best_model_path)

lstm_test_results = lstm_trainer.test(
    dataloaders=lstm_test_loader,
    ckpt_path=best_model_path
)
print("\nResultados do Teste do Melhor Modelo LSTM:", lstm_test_results)

### GRU

In [ ]:
# Data Preparation for GRU
# Converter features para tensores 3D [batch, seq_len, features]
X_train_gru = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)  # (N, 140, 1)
X_val_gru   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1)
X_test_gru  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)

print("Tensores prontos para GRU:")
print("X_train_gru:", X_train_gru.shape)
print("X_val_gru:", X_val_gru.shape)
print("X_test_gru:", X_test_gru.shape)
print("y_train:", y_train.shape)

# DataLoaders
gru_train_dataset = TensorDataset(X_train_gru, y_train)
gru_val_dataset   = TensorDataset(X_val_gru, y_val)
gru_test_dataset  = TensorDataset(X_test_gru, y_test)

gru_train_loader = DataLoader(gru_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
gru_val_loader   = DataLoader(gru_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
gru_test_loader  = DataLoader(gru_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Exemplo de batch
gru_batch_X, gru_batch_y = next(iter(gru_train_loader))
print("\nExemplo de batch GRU:")
print("gru_batch_X:", gru_batch_X.shape)  # (batch_size, seq_len, features)
print("gru_batch_y:", gru_batch_y.shape)  # (batch_size,)


In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size=1, num_classes=5):
        super(GRUModel, self).__init__()

        hidden_size = 64 

        # Camada GRU 
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        # Classificador 
        self.fc1 = nn.Linear(hidden_size, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, num_classes)

        # Inicialização de pesos
        nn.init.kaiming_normal_(self.fc1.weight)
        nn.init.kaiming_normal_(self.fc2.weight)
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        out, h_n = self.gru(x)
        last_hidden = h_n[-1]

        x = F.relu(self.fc1(last_hidden))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
class LitGRU(pl.LightningModule):
    def __init__(self, input_size=1, num_classes=5, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.criterion = nn.CrossEntropyLoss()
        self.model = GRUModel(input_size=input_size, num_classes=num_classes)
    
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("val_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("val_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log("test_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("test_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


# EarlyStopping
gru_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    verbose=True,
    mode="min"
)

# ModelCheckpoint
gru_checkpoint = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-gru-model",
    verbose=True
)

In [ ]:
# Instanciar modelo e logger 
gru_model = LitGRU(input_size=1, num_classes=NUM_CLASSES, lr=LEARNING_RATE)
gru_metrics_logger = MetricsLogger()

# Treiner
gru_trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="cpu",
    log_every_n_steps=10,
    deterministic=True,
    callbacks=[gru_early_stop, gru_metrics_logger, gru_checkpoint]
)

# Trainar
gru_trainer.fit(gru_model, train_dataloaders=gru_train_loader, val_dataloaders=gru_val_loader)

In [ ]:
# Métricas em DataFrame
gru_df_metrics = metrics_df(gru_metrics_logger)
display(gru_df_metrics)

# Guardar CSV
save_metrics_csv(gru_metrics_logger, filename="defaultModelGRU.csv", folder=RESULTS_FOLDER)

In [ ]:
# Teste final com o melhor modelo
best_model_path = gru_checkpoint.best_model_path
print("Best GRU model saved at:", best_model_path)

gru_test_results = gru_trainer.test(
    dataloaders=gru_test_loader,
    ckpt_path=best_model_path
)
print("\nResultados do Teste do Melhor Modelo GRU:", gru_test_results)

## Resultados

Os testes finais desta execução estão apresentados na conclusão. Os gráficos abaixo leem os CSV disponíveis em `results/ecg5000/`, incluindo ficheiros históricos; as curvas mostram treino e validação, não o conjunto de teste.


### CNN 1D

O código deste notebook treina um modelo CNN 1D. Os CSV em `results/ecg5000/` incluem execuções recuperadas do trabalho original. Os nomes dos ficheiros não identificam de forma suficiente todas as variantes descritas no texto original; por isso, os gráficos apresentam apenas as métricas registadas, sem atribuir configurações não verificadas.


### LSTM

O código deste notebook treina um modelo LSTM. Os CSV em `results/ecg5000/` incluem execuções recuperadas do trabalho original. Os nomes dos ficheiros não identificam de forma suficiente todas as variantes descritas no texto original; por isso, os gráficos apresentam apenas as métricas registadas, sem atribuir configurações não verificadas.


### GRU

O código deste notebook treina um modelo GRU. Os CSV em `results/ecg5000/` incluem execuções recuperadas do trabalho original. Os nomes dos ficheiros não identificam de forma suficiente todas as variantes descritas no texto original; por isso, os gráficos apresentam apenas as métricas registadas, sem atribuir configurações não verificadas.


In [ ]:
def plot_results(folder_path=RESULTS_FOLDER):
    files = sorted(Path(folder_path).glob("*.csv"))
    if not files:
        print("Sem CSV para representar. Executa os treinos primeiro.")
        return

    for model_name in ("CNN", "LSTM", "GRU"):
        group = [file for file in files if model_name.lower() in file.stem.lower()]
        if not group:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for file in group:
            df = pd.read_csv(file)
            for ax, key, label in zip(axes, ("loss", "acc"), ("Perda", "Exatidão")):
                ax.plot(df["epoch"], df[f"train_{key}"], label=f"{file.stem} · treino")
                ax.plot(df["epoch"], df[f"val_{key}"], label=f"{file.stem} · val.")
                ax.set(title=label, xlabel="Época", ylabel=label)
                ax.grid(True)
        for ax in axes:
            ax.legend(fontsize=7)
        fig.suptitle(f"{model_name} · métricas dos CSV disponíveis")
        fig.tight_layout()
        plt.show()

plot_results()


## Conclusões

Na execução atual, a CNN obteve **94,4%** de exatidão no conjunto de teste (`test_acc_epoch = 0,944`), enquanto a LSTM e a GRU obtiveram **93,6%** cada (`0,936`). As perdas de teste registadas foram **0,227** para a CNN, **0,260** para a LSTM e **0,233** para a GRU. Neste ensaio, a CNN apresentou a maior exatidão; a diferença é pequena e uma única execução não permite concluir que esta arquitetura é geralmente superior.

Os gráficos também incluem CSV recuperados do trabalho original. Esses ficheiros correspondem a execuções anteriores com números de épocas diferentes e sem registo completo das configurações. Não devem ser confundidos com os resultados de teste da execução atual nem usados para ordenar as arquiteturas de forma definitiva. O objetivo deste estudo é comparar o comportamento das três abordagens num exercício académico, sem tirar conclusões clínicas.
